# Lab 2 — Ứng dụng Pre-trained Models trên CIFAR-10

> **Môn học:** Deep Learning — **Chủ đề:** Transfer Learning & Fine-tuning bằng PyTorch  
> **Tập dữ liệu:** CIFAR-10 (60.000 ảnh màu 32×32, 10 lớp: 45.000 Train / 5.000 Validation / 10.000 Test)

Notebook này điều phối toàn bộ quy trình thực nghiệm phân loại ảnh CIFAR-10 bằng các mô hình **VGG16** và **ResNet-18** (trọng số `IMAGENET1K_V1`) kết hợp cùng gói thư viện mô-đun hóa **`going_modular/`**. Mỗi ô mã (cell) được thiết kế theo nguyên tắc **đơn nhiệm (Single Responsibility)**, ngắn gọn, dễ theo dõi và gỡ lỗi.

## Bản tóm tắt cấu trúc Notebook

| Mục | Giai đoạn | Các bước triển khai theo từng Cell |
| :--- | :--- | :--- |
| **0** | **Setup & Run Configuration** | `0.1a` Đồng bộ Git/Colab → `0.1b` Thiết lập đường dẫn → `0.2a` Import thư viện → `0.2b` Kiểm tra thiết bị → `0.3a` Khai báo `RUN_SPECS` → `0.3b` Hàm kiểm tra cấu hình → `0.4` Bảng điều khiển Run |
| **1** | **CIFAR-10 Data Pipeline** | `1.1a` Hàm xác thực/giải nén dữ liệu → `1.1b` Thực thi stage dữ liệu → `1.2a` Hàm tạo DataLoader → `1.2b` Kiểm tra Batch mẫu → `1.3` Hiển thị 16 ảnh mẫu |
| **2** | **Pretrained Architectures** | `2.1` Hàm khảo sát & Cấu trúc **ResNet-18** → `2.2` Khảo sát cấu trúc **VGG16** |
| **3** | **CIFAR-10 Model Adaptation** | `3.1` Hàm dựng mô hình thích nghi cho từng chế độ (`build_model_for_run`) |
| **4** | **Training & Validation** | `4.1a` Đường dẫn Artifact → `4.1b` Kiểm tra Action → `4.1c` Optimizer & BatchNorm hook → `4.2a` Nhánh Train/Resume → `4.2b` Nhánh Reuse → `4.2c` Hàm chạy 1 Run → `4.2d` Vòng lặp `RUN_ID_LIST` |
| **5** | **Model Comparison** | `5.1` Bảng tổng hợp Validation & Tham số → `5.2` Biểu đồ đường cong Loss & Accuracy |
| **6** | **Final Test Evaluation** | `6.1a` Suy luận 1 lượt trên Test → `6.1b` Đo độ trễ & Lưu `test_metrics.json` → `6.1c` Bảng đối sánh Markdown → `6.2` Confusion Matrix → `6.3a` Per-class Accuracy → `6.3b` Top 5 nhầm lẫn → `6.4` Trực quan hóa ảnh đoán sai |

---
## 0. Thiết lập Môi trường & Cấu hình Điều phối (Setup & Run Configuration)

Mục này được chia thành các cell nhỏ theo từng chức năng độc lập:
- **0.1a & 0.1b:** Nhận diện môi trường (Colab/Local), đồng bộ mã nguồn từ GitHub và thiết lập các biến đường dẫn (`CODE_ROOT`, `DATA_DIR`, `ARTIFACT_ROOT`, `OUTPUT_ROOT`, `LOG_ROOT`).
- **0.2a & 0.2b:** Nạp thư viện và kiểm tra thiết bị phần cứng (`DEVICE`, `ENV_INFO`).
- **0.3a & 0.3b:** Định nghĩa danh mục cấu hình `RUN_SPECS` và các hàm kiểm tra tính hợp lệ của danh sách thí nghiệm.
- **0.4:** Bảng điều khiển dành cho người dùng (`RUN_ID`, `RUN_IDS`, `RUN_PLAN`, `FORCE_RETRAIN`).

In [ ]:
# -- 0.1a. Nhận diện môi trường (Colab / Local) & Đồng bộ GitHub repository --
from pathlib import Path
import subprocess
import sys

GITHUB_REPO_URL = "https://github.com/toan58936/lab2.git"
GITHUB_BRANCH = "main"
GITHUB_COMMIT = None  # Điền commit hash nếu muốn cố định phiên bản mã nguồn

try:
    import google.colab  # type: ignore
    IS_COLAB = True
except ModuleNotFoundError:
    IS_COLAB = False

if IS_COLAB:
    from google.colab import drive  # type: ignore
    drive.mount("/content/drive")

    CODE_ROOT = Path("/content/lab2")
    if not CODE_ROOT.exists():
        subprocess.run(
            ["git", "clone", "--branch", GITHUB_BRANCH, "--single-branch", GITHUB_REPO_URL, str(CODE_ROOT)],
            check=True,
        )
    elif not (CODE_ROOT / ".git").exists():
        raise FileNotFoundError(f"{CODE_ROOT} đã tồn tại nhưng không phải Git checkout.")

    if GITHUB_COMMIT is not None:
        subprocess.run(["git", "-C", str(CODE_ROOT), "checkout", GITHUB_COMMIT], check=True)

    CODE_REVISION = subprocess.run(
        ["git", "-C", str(CODE_ROOT), "rev-parse", "HEAD"],
        check=True, capture_output=True, text=True,
    ).stdout.strip()
else:
    CODE_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    CODE_REVISION = "local working tree"

CODE_ROOT = CODE_ROOT.resolve()
if str(CODE_ROOT) not in sys.path:
    sys.path.insert(0, str(CODE_ROOT))

print(f"Environment   : {'Google Colab' if IS_COLAB else 'Local Machine'}")
print(f"CODE_ROOT     : {CODE_ROOT}")
print(f"CODE_REVISION : {CODE_REVISION}")

In [ ]:
# -- 0.1b. Thiết lập các đường dẫn dữ liệu, artifacts, outputs và logs --
if IS_COLAB:
    DRIVE_ROOT = Path("/content/drive/MyDrive/lab2")
    DRIVE_DATA_ROOT = DRIVE_ROOT / "data"
    DATA_DIR = Path("/content/cifar10_data")
    ARTIFACT_ROOT = DRIVE_ROOT / "artifacts"
    OUTPUT_ROOT = DRIVE_ROOT / "outputs"
    LOG_ROOT = DRIVE_ROOT / "logs"
    RUNS_ROOT = DRIVE_ROOT / "runs"
else:
    DRIVE_ROOT = None
    DRIVE_DATA_ROOT = None
    DATA_DIR = CODE_ROOT / "data"
    ARTIFACT_ROOT = CODE_ROOT / "artifacts"
    OUTPUT_ROOT = CODE_ROOT / "outputs"
    LOG_ROOT = CODE_ROOT / "logs"
    RUNS_ROOT = CODE_ROOT / "runs"

print(f"DRIVE_DATA_ROOT : {DRIVE_DATA_ROOT}")
print(f"DATA_DIR        : {DATA_DIR}")
print(f"ARTIFACT_ROOT   : {ARTIFACT_ROOT}")
print(f"OUTPUT_ROOT     : {OUTPUT_ROOT}")
print(f"LOG_ROOT        : {LOG_ROOT}")

In [ ]:
# -- 0.2a. Nạp thư viện chuẩn, PyTorch và các mô-đun từ going_modular --
try:
    from torchinfo import summary as torchinfo_summary
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torchinfo"], check=True)
    from torchinfo import summary as torchinfo_summary

from functools import partial
from timeit import default_timer as timer
import matplotlib.pyplot as plt
import pandas as pd
import torch
from torch import nn
import torchvision.models as models

try:
    from torch.utils.tensorboard import SummaryWriter
except ImportError as exc:
    raise ImportError("Cần cài đặt tensorboard: pip install tensorboard") from exc

from going_modular import data_setup, engine, evaluate, model_builder, utils
from going_modular.model_builder import (
    ModelConfig,
    build_model,
    count_params,
    get_param_groups,
    set_train_mode,
)
from going_modular.evaluate import (
    compute_confusion_matrix,
    denormalize as eval_denormalize,
    evaluate_predictions,
    run_inference,
)

In [ ]:
# -- 0.2b. Khởi tạo thiết bị tính toán (DEVICE) & Kiểm tra phiên bản thư viện --
DEVICE = utils.get_device()
ENV_INFO = utils.get_env_info(DEVICE)

for key, val in ENV_INFO.items():
    print(f"{key:<15} : {val}")
print(f"{'cuda_available':<15} : {torch.cuda.is_available()}")

In [ ]:
# -- 0.3a. Khai báo hằng số toàn cục & Danh mục cấu hình thí nghiệm (RUN_SPECS) --
SEED = 42
VAL_SIZE = 5_000
NUM_WORKERS = 0
TRANSFORM_WEIGHTS = "IMAGENET1K_V1"

RUN_SPECS = {
    "v0_vgg16": {
        "arch": "vgg16",
        "weights": "IMAGENET1K_V1",
        "freeze_backbone": True,
        "unfreeze_last_n": 0,
        "dropout": 0.0,
        "freeze_bn_stats": True,
        "epochs": 3,
        "start_epoch": 0,
        "batch_size": 32,
        "lr": 1e-3,
        "optimizer": "adam",
        "param_groups": None,
        "dependency_run_id": None,
        "source_checkpoint": None,
    },
    "v1_resnet18": {
        "arch": "resnet18",
        "weights": "IMAGENET1K_V1",
        "freeze_backbone": True,
        "unfreeze_last_n": 0,
        "dropout": 0.0,
        "freeze_bn_stats": True,
        "epochs": 3,
        "start_epoch": 0,
        "batch_size": 32,
        "lr": 1e-3,
        "optimizer": "adam",
        "param_groups": None,
        "dependency_run_id": None,
        "source_checkpoint": None,
    },
    "v1_resnet18_finetune": {
        "arch": "resnet18",
        "weights": None,
        "freeze_backbone": True,
        "unfreeze_last_n": 1,
        "dropout": 0.0,
        "freeze_bn_stats": True,
        "epochs": 3,
        "start_epoch": 3,
        "batch_size": 32,
        "lr": 1e-4,
        "optimizer": "adam",
        "param_groups": {"head": 1e-3, "backbone": 1e-4},
        "dependency_run_id": "v1_resnet18",
        "source_checkpoint": "v1_resnet18/best_model.pth",
    },
}

In [ ]:
# -- 0.3b. Các hàm chuẩn hóa danh sách RUN_IDS và kiểm tra ràng buộc phụ thuộc --
def _normalize_run_ids(run_id, run_ids):
    if run_ids is not None:
        items = list(run_ids)
    elif isinstance(run_id, (list, tuple)):
        items = list(run_id)
    elif isinstance(run_id, str) and "," in run_id:
        items = [s.strip() for s in run_id.split(",") if s.strip()]
    else:
        items = [run_id]

    if not items:
        raise ValueError("Danh sách RUN_ID không được rỗng.")

    allowed = tuple(RUN_SPECS)
    normalized, seen = [], set()
    for rid in items:
        if rid not in RUN_SPECS:
            raise ValueError(f"Unknown RUN_ID {rid!r}; choose one of {allowed}.")
        if rid in seen:
            raise ValueError(f"Duplicate RUN_ID in list: {rid!r}.")
        seen.add(rid)
        normalized.append(rid)

    for rid in normalized:
        dep = RUN_SPECS[rid].get("dependency_run_id")
        if dep and dep in normalized and normalized.index(dep) > normalized.index(rid):
            raise ValueError(f"Run phụ thuộc '{dep}' phải được chạy trước '{rid}'.")
    return normalized


def _get_run_action(run_id):
    action = RUN_PLAN.get(run_id, DEFAULT_RUN_ACTION)
    if action not in {"train", "resume", "reuse"}:
        raise ValueError(
            f"Action không hợp lệ cho {run_id}: {action!r}. Chọn 'train', 'resume', hoặc 'reuse'."
        )
    return action

In [ ]:
# -- 0.4. BẢNG ĐIỀU KHIỂN RUN (Người dùng chọn run và chế độ chạy tại đây) --
RUN_ID = "v0_vgg16"
RUN_IDS = None  # Ví dụ: ["v0_vgg16", "v1_resnet18", "v1_resnet18_finetune"]

DEFAULT_RUN_ACTION = "train"  # "train" | "resume" | "reuse"
RUN_PLAN = {
    # "v0_vgg16": "train",
    # "v1_resnet18": "resume",
    # "v1_resnet18_finetune": "reuse",
}
FORCE_RETRAIN = False

RUN_ID_LIST = _normalize_run_ids(RUN_ID, RUN_IDS)
run_results = {}

print(f"RUN_IDS         : {RUN_ID_LIST}")
print(f"DEFAULT_ACTION  : {DEFAULT_RUN_ACTION} | FORCE_RETRAIN: {FORCE_RETRAIN}")
print(f"Global Params   : Seed={SEED} | Val_size={VAL_SIZE} | Num_workers={NUM_WORKERS}")
print("-" * 80)
for rid in RUN_ID_LIST:
    s = RUN_SPECS[rid]
    print(
        f"  - {rid:<20} : action={_get_run_action(rid):<6} | arch={s['arch']:<8} | "
        f"unfreeze={s['unfreeze_last_n']} | epochs={s['epochs']} | bs={s['batch_size']} | lr={s['lr']}"
    )

---
## 1. Chuẩn bị Dữ liệu CIFAR-10 (Data Staging & DataLoaders)

Mục này được tách thành 5 cell theo đúng luồng xử lý dữ liệu:
- **1.1a:** Định nghĩa hàm kiểm tra tính toàn vẹn và giải nén an toàn dữ liệu CIFAR-10 từ Google Drive.
- **1.1b:** Thực thi đồng bộ dữ liệu vào ổ đĩa Runtime (`DATA_DIR`).
- **1.2a:** Định nghĩa hàm tạo DataLoader (`create_dataloaders_for_spec`) với bộ sinh số ngẫu nhiên độc lập.
- **1.2b:** Khởi tạo DataLoader cho run đầu tiên và kiểm tra kích thước batch, nhãn lớp, miền giá trị pixel.
- **1.3:** Giải chuẩn hóa (de-normalize) và trực quan hóa lưới 16 ảnh mẫu.

In [ ]:
# -- 1.1a. Hàm xác thực thư mục CIFAR-10 và sao chép/giải nén an toàn từ Drive --
import shutil
import tarfile
import tempfile
from pathlib import PurePosixPath

_CIFAR_REQUIRED_FILES = {
    "data_batch_1", "data_batch_2", "data_batch_3", "data_batch_4",
    "data_batch_5", "test_batch", "batches.meta",
}


def _valid_cifar10_directory(path: Path) -> bool:
    return path.is_dir() and _CIFAR_REQUIRED_FILES.issubset(
        {item.name for item in path.iterdir() if item.is_file()}
    )


def _stage_cifar10_from_drive() -> None:
    destination = DATA_DIR / "cifar-10-batches-py"
    if _valid_cifar10_directory(destination):
        return
    if destination.exists():
        raise FileExistsError(f"Thư mục CIFAR-10 tại {destination} không đầy đủ; dừng để tránh ghi đè.")

    source_directory = DRIVE_DATA_ROOT / "cifar-10-batches-py"
    if not _valid_cifar10_directory(source_directory) and _valid_cifar10_directory(DRIVE_DATA_ROOT):
        source_directory = DRIVE_DATA_ROOT

    DATA_DIR.mkdir(parents=True, exist_ok=True)
    if _valid_cifar10_directory(source_directory):
        shutil.copytree(source_directory, destination)
        return

    archive_path = DRIVE_DATA_ROOT / "cifar-10-python.tar.gz"
    if not archive_path.is_file():
        raise FileNotFoundError(f"Không tìm thấy CIFAR-10 trên Drive ({source_directory} hoặc {archive_path}).")

    with tempfile.TemporaryDirectory(prefix="cifar10_stage_") as tmp_dir:
        with tarfile.open(archive_path, "r:gz") as archive:
            members = archive.getmembers()
            for member in members:
                member_path = PurePosixPath(member.name)
                if (
                    member_path.is_absolute()
                    or ".." in member_path.parts
                    or not member_path.parts
                    or member_path.parts[0] != "cifar-10-batches-py"
                    or not (member.isfile() or member.isdir())
                ):
                    raise ValueError(f"Thành phần không an toàn trong file nén: {member.name}")
            archive.extractall(tmp_dir, members=members)
        extracted_dir = Path(tmp_dir) / "cifar-10-batches-py"
        if not _valid_cifar10_directory(extracted_dir):
            raise ValueError(f"File nén không chứa bộ dữ liệu CIFAR-10 hợp lệ: {archive_path}")
        shutil.copytree(extracted_dir, destination)

In [ ]:
# -- 1.1b. Thực thi kiểm tra và đồng bộ dữ liệu CIFAR-10 vào Runtime --
if IS_COLAB:
    _stage_cifar10_from_drive()
elif not _valid_cifar10_directory(DATA_DIR / "cifar-10-batches-py"):
    raise FileNotFoundError(
        f"Không tìm thấy dữ liệu CIFAR-10 dưới {DATA_DIR}; hãy chuẩn bị dữ liệu trước khi chạy."
    )

print(f"CIFAR-10 dataset verified at: {DATA_DIR / 'cifar-10-batches-py'}")

In [ ]:
# -- 1.2a. Định nghĩa hàm tạo DataLoaders theo cấu hình của từng Run --
def create_dataloaders_for_spec(spec: dict):
    """Tạo (train_dl, val_dl, test_dl, CLASS_NAMES) với generator độc lập theo SEED."""
    utils.set_seed(SEED)
    train_generator = torch.Generator().manual_seed(SEED)
    return data_setup.create_train_val_test_dataloaders(
        batch_size=spec["batch_size"],
        data_dir=str(DATA_DIR),
        transform_model=spec["arch"],
        transform_weights=TRANSFORM_WEIGHTS,
        val_size=VAL_SIZE,
        seed=SEED,
        num_workers=NUM_WORKERS,
        generator=train_generator,
        download=False,
    )

In [ ]:
# -- 1.2b. Kiểm tra Batch dữ liệu mẫu cho Run đầu tiên trong danh sách --
_first_run_id = RUN_ID_LIST[0]
_first_spec = RUN_SPECS[_first_run_id]
train_dl, val_dl, test_dl, CLASS_NAMES = create_dataloaders_for_spec(_first_spec)
NUM_CLASSES = len(CLASS_NAMES)

images, labels = next(iter(train_dl))
print(f"Run inspect     : {_first_run_id} (arch={_first_spec['arch']}, batch_size={_first_spec['batch_size']})")
print(f"Classes ({NUM_CLASSES})    : {CLASS_NAMES}")
print(f"Train loader    : {len(train_dl):>5} batches ({len(train_dl.dataset):,} samples)")
print(f"Val loader      : {len(val_dl):>5} batches ({len(val_dl.dataset):,} samples)")
print(f"Test loader     : {len(test_dl):>5} batches ({len(test_dl.dataset):,} samples)")
print(f"Batch images    : {images.shape} (dtype={images.dtype})")
print(f"Batch labels    : {labels.shape}")
print(f"Pixel range     : [{images.min():.3f}, {images.max():.3f}]")

In [ ]:
# -- 1.3. Giải chuẩn hóa (De-normalize) và hiển thị 16 ảnh mẫu tập Train --
mean_vals, std_vals = data_setup.get_pretrained_mean_std(
    model_name=_first_spec["arch"],
    weights_name=TRANSFORM_WEIGHTS,
)

fig, axes = plt.subplots(4, 4, figsize=(10, 10))
for ax, img, lbl in zip(axes.flat, images[:16], labels[:16]):
    vis_img = eval_denormalize(img, mean_vals, std_vals)
    ax.imshow(vis_img.permute(1, 2, 0).numpy())
    ax.set_title(CLASS_NAMES[lbl.item()], fontsize=9)
    ax.axis("off")

fig.suptitle("16 ảnh mẫu tập Train (De-normalized, Center Crop 224×224)", fontsize=13)
plt.tight_layout()
plt.show()

---
## 2. Khám phá Kiến trúc Pre-trained (VGG16 & ResNet-18)

Tách riêng việc khảo sát từng họ kiến trúc thành 2 cell độc lập để dễ quan sát bảng `torchinfo.summary` và đối chiếu số lượng tham số:
- **2.1:** Định nghĩa hàm khảo sát & Hiển thị cấu trúc **ResNet-18** (`total = 11,181,642`, `trainable = 5,130`).
- **2.2:** Hiển thị cấu trúc **VGG16** (`total = 134,301,514`, `trainable = 40,970`).

In [ ]:
# -- 2.1. Hàm tiện ích khảo sát kiến trúc & Thống kê chi tiết mô hình ResNet-18 --
def inspect_architecture(arch_name: str) -> None:
    utils.set_seed(SEED)
    cfg_explore = ModelConfig(
        arch=arch_name,
        num_classes=NUM_CLASSES,
        weights=None,
        freeze_backbone=True,
    )
    model_explore = build_model(cfg_explore)
    param_stats = count_params(model_explore)
    head_module = model_explore.fc if arch_name == "resnet18" else model_explore.classifier[6]

    print("=" * 75)
    print(f"Kiến trúc: {arch_name.upper()} | num_classes={NUM_CLASSES} | freeze_backbone=True")
    print(
        f"Tham số  : Total={param_stats['total']:,} | "
        f"Trainable={param_stats['trainable']:,} | Frozen={param_stats['frozen']:,}"
    )
    print(f"Head mới : {head_module}")
    print("=" * 75)
    print(
        torchinfo_summary(
            model=model_explore,
            input_size=(1, 3, 224, 224),
            col_names=["input_size", "output_size", "num_params", "trainable"],
            col_width=20,
            row_settings=["var_names"],
            depth=3,
        )
    )


inspect_architecture("resnet18")

In [ ]:
# -- 2.2. Thống kê chi tiết mô hình VGG16 (Native G1: thay classifier[6]) --
inspect_architecture("vgg16")

---
## 3. Thích nghi Mô hình cho CIFAR-10 (Model Adaptation & BatchNorm Policy)

Định nghĩa hàm `build_model_for_run` để khởi tạo mô hình Torchvision gốc theo từng chế độ:
- **`action="train"` (mới):** Tải trọng số `IMAGENET1K_V1` cho Feature Extraction (`v0_vgg16`, `v1_resnet18`); hoặc nạp `v1_resnet18/best_model.pth` rồi mở khóa `layer4` cho Fine-tuning (`v1_resnet18_finetune`).
- **`action="resume"` / `"reuse"`:** Dựng khung kiến trúc (`weights=None`) để nạp trạng thái từ checkpoint của chính run đó.

In [ ]:
# -- 3.1. Hàm dựng model và ModelConfig cho từng Run --
def build_model_for_run(run_id: str, spec: dict, action: str, resume_flag: bool = False):
    load_weights = spec["weights"] if (action == "train" and not resume_flag) else None

    cfg = ModelConfig(
        arch=spec["arch"],
        num_classes=NUM_CLASSES,
        weights=load_weights,
        freeze_backbone=spec["freeze_backbone"],
        unfreeze_last_n=spec["unfreeze_last_n"],
        dropout=spec.get("dropout", 0.0),
        freeze_bn_stats=spec.get("freeze_bn_stats", True),
    )
    model = build_model(cfg).to(DEVICE)

    if action == "train" and not resume_flag and spec.get("dependency_run_id") is not None:
        baseline_path = ARTIFACT_ROOT / spec["source_checkpoint"]
        if not baseline_path.is_file():
            raise FileNotFoundError(
                f"Run '{run_id}' yêu cầu baseline checkpoint tại: {baseline_path}. "
                f"Hãy hoàn tất run baseline '{spec['dependency_run_id']}' trước."
            )
        utils.load_model(model, str(baseline_path), device=str(DEVICE), arch=spec["arch"])
        print(f"  [Init] Đã nạp baseline checkpoint cho fine-tune: {baseline_path}")

    if action in {"resume", "reuse"} and spec.get("dependency_run_id") is not None:
        print(f"  [Init] {run_id}: khởi tạo khung kiến trúc để nạp trạng thái của chính run ({action}).")

    return model, cfg

---
## 4. Huấn luyện & Kiểm định Tuần tự (Training, Validation & Orchestration)

Mục 4 được chia thành **7 cell đơn nhiệm** theo đúng luồng điều phối:
- **4.1a:** Khởi tạo hàm mất mát (`CrossEntropyLoss`), quản lý đường dẫn Artifacts và hàm ghi cấu hình lên TensorBoard.
- **4.1b:** Hàm kiểm tra chốt chặn an toàn (`_validate_action`) cho 3 chế độ `train`, `resume`, `reuse`.
- **4.1c:** Hàm tạo Optimizer (hỗ trợ 1 hoặc 2 nhóm Learning Rate) và callback giữ BatchNorm đóng băng ở `eval()`.
- **4.2a:** Hàm xử lý nhánh huấn luyện mới hoặc tiếp tục từ checkpoint (`_run_train_or_resume`).
- **4.2b:** Hàm xử lý nhánh tái sử dụng kết quả đã lưu (`_run_reuse`).
- **4.2c:** Hàm điều phối cấp cao cho một thí nghiệm đơn lẻ (`execute_single_run`).
- **4.2d:** Vòng lặp chính chạy tuần tự qua `RUN_ID_LIST` và lưu vào `run_results`.

In [ ]:
# -- 4.1a. Khởi tạo hàm mất mát, quản lý đường dẫn Artifacts & log cấu hình TensorBoard --
loss_fn = nn.CrossEntropyLoss()


def log_run_config(writer: SummaryWriter, config: dict) -> None:
    """Ghi cấu hình run vào tab Text của TensorBoard."""
    lines = [f"`{k}` = `{v}`" for k, v in config.items()]
    writer.add_text("run_config", "  \n".join(lines), global_step=0)


def _prepare_artifact_paths(run_id: str) -> dict:
    """Tạo thư mục artifacts và trả về đường dẫn các file chuẩn của run."""
    dir_path = ARTIFACT_ROOT / run_id
    dir_path.mkdir(parents=True, exist_ok=True)
    return {
        "dir": dir_path,
        "config": dir_path / "config.json",
        "history": dir_path / "history.json",
        "best_model": dir_path / "best_model.pth",
        "last_checkpoint": dir_path / "last_checkpoint.pth",
        "test_metrics": dir_path / "test_metrics.json",
    }


def _artifact_files_present(paths: dict) -> bool:
    return all(p.is_file() for p in [paths["config"], paths["history"], paths["best_model"]])

In [ ]:
# -- 4.1b. Hàm kiểm tra tính hợp lệ của Action và bảo vệ Artifacts hiện có --
def _validate_action(run_id: str, spec: dict, action: str, paths: dict, force_retrain: bool) -> bool:
    """Xác thực trạng thái artifacts theo action ('train' | 'resume' | 'reuse'). Trả về resume_flag."""
    if action == "train":
        if _artifact_files_present(paths):
            if not force_retrain:
                raise FileExistsError(
                    f"{run_id}: artifacts đã tồn tại tại {paths['dir']}. "
                    "Đặt FORCE_RETRAIN=True để xóa và train lại, hoặc chọn action='resume'/'reuse'."
                )
            print(f"[FORCE_RETRAIN] {run_id}: xóa artifacts cũ và train lại từ đầu...")
            for f in [
                paths["config"],
                paths["history"],
                paths["best_model"],
                paths["last_checkpoint"],
                paths["test_metrics"],
            ]:
                if f.is_file():
                    f.unlink()
        return False

    if action == "resume":
        if not paths["last_checkpoint"].is_file():
            raise FileNotFoundError(
                f"{run_id}: action='resume' yêu cầu {paths['last_checkpoint']}. Hãy chọn action='train' nếu chạy mới."
            )
        _ckpt = torch.load(str(paths["last_checkpoint"]), map_location="cpu", weights_only=True)
        done_epochs = int(_ckpt.get("phase_epochs_completed", 0))
        target_epochs = int(spec["epochs"])
        if done_epochs >= target_epochs:
            raise ValueError(
                f"{run_id}: đã hoàn tất {done_epochs}/{target_epochs} epochs. Hãy chọn action='reuse'."
            )
        print(f"  [Resume] {run_id}: tiếp tục từ epoch {done_epochs}/{target_epochs}")
        return True

    if action == "reuse":
        if not _artifact_files_present(paths):
            missing = [
                str(p) for p in [paths["config"], paths["history"], paths["best_model"]] if not p.is_file()
            ]
            raise FileNotFoundError(f"{run_id}: thiếu artifacts cho chế độ reuse: {missing}.")
        return False

    raise ValueError(f"Action không được hỗ trợ: {action!r}")

In [ ]:
# -- 4.1c. Hàm khởi tạo Optimizer (1 hoặc 2 nhóm LR) và chính sách BatchNorm --
def _build_optimizer(model_selected: nn.Module, cfg_selected: ModelConfig, spec: dict, lr: float):
    if spec["param_groups"] is not None:
        param_groups = get_param_groups(
            model_selected,
            cfg_selected,
            lr_head=spec["param_groups"]["head"],
            lr_backbone=spec["param_groups"]["backbone"],
        )
        return torch.optim.Adam(param_groups, lr=lr)
    return torch.optim.Adam(
        filter(lambda p: p.requires_grad, model_selected.parameters()),
        lr=lr,
    )


def _build_train_mode_fn(cfg_selected: ModelConfig, spec: dict):
    if spec["freeze_backbone"]:
        return partial(set_train_mode, cfg=cfg_selected)
    return None

In [ ]:
# -- 4.2a. Hàm xử lý nhánh Huấn luyện mới hoặc Tiếp tục từ checkpoint (_run_train_or_resume) --
def _run_train_or_resume(
    run_id: str, spec: dict, action: str, resume_flag: bool,
    model_selected: nn.Module, cfg_selected: ModelConfig,
    train_dl, val_dl, paths: dict, session_id: str
):
    optimizer_selected = _build_optimizer(model_selected, cfg_selected, spec, spec["lr"])
    train_mode_fn = _build_train_mode_fn(cfg_selected, spec)
    session_log_dir = utils.make_session_log_dir(LOG_ROOT, run_id, session_id)
    writer_selected = SummaryWriter(log_dir=str(session_log_dir))

    phase_config = {
        "run_id": run_id,
        "arch": spec["arch"],
        "model_config": cfg_selected.to_dict(),
        "data": {"transform_model": spec["arch"], "transform_weights": TRANSFORM_WEIGHTS, "val_size": VAL_SIZE},
        "batch_size": spec["batch_size"],
        "lr": spec["lr"],
        "param_groups": spec["param_groups"],
        "epochs": spec["epochs"],
        "start_epoch": spec.get("start_epoch", 0),
        "seed": SEED,
    }
    run_config = {
        **phase_config,
        "action": action,
        "resume": resume_flag,
        "session_id": session_id,
        "env_info": ENV_INFO,
    }

    utils.save_config(str(paths["dir"]), run_config)
    log_run_config(writer_selected, run_config)

    print("-" * 68)
    print(f"Run: {run_id} | Action: {action} (resume={resume_flag}) | Target Epochs: {spec['epochs']}")
    print("-" * 68)

    start_time = timer()
    try:
        history_selected = engine.train(
            model=model_selected,
            train_dataloader=train_dl,
            val_dataloader=val_dl,
            optimizer=optimizer_selected,
            loss_fn=loss_fn,
            epochs=spec["epochs"],
            device=DEVICE,
            train_mode_fn=train_mode_fn,
            writer=writer_selected,
            start_epoch=spec.get("start_epoch", 0),
            save_path=str(paths["best_model"]),
            checkpoint_dir=str(paths["dir"]),
            resume=resume_flag,
            expected_phase_config=phase_config,
            verbose=True,
        )
    finally:
        writer_selected.close()

    train_time = sum(history_selected.get("epoch_time", [])) or (timer() - start_time)
    run_config["train_seconds"] = train_time
    utils.save_config(str(paths["dir"]), run_config)
    utils.save_history(str(paths["dir"]), history_selected)
    best_val_acc = max(history_selected["val_acc"])
    print(f"=> {run_id} hoàn tất | action={action} | train_time={train_time:.1f}s | best_val_acc={best_val_acc:.4f}")
    return history_selected, best_val_acc, train_time

In [ ]:
# -- 4.2b. Hàm xử lý nhánh Tái sử dụng kết quả đã lưu (_run_reuse) --
def _run_reuse(run_id: str, spec: dict, model_selected: nn.Module, paths: dict):
    history_selected, saved_cfg = utils.load_experiment(str(paths["dir"]))
    utils.load_model(model_selected, str(paths["best_model"]), device=str(DEVICE), arch=spec["arch"])
    train_time = float(sum(history_selected.get("epoch_time", [])) or saved_cfg.get("train_seconds", 0.0))
    best_val_acc = max(history_selected["val_acc"]) if history_selected.get("val_acc") else 0.0
    print(f"=> {run_id} reused | train_time={train_time:.1f}s | best_val_acc={best_val_acc:.4f}")
    return history_selected, best_val_acc, train_time

In [ ]:
# -- 4.2c. Hàm điều phối thực thi một Run đơn lẻ (execute_single_run) --
def execute_single_run(run_id: str) -> dict:
    spec = RUN_SPECS[run_id]
    action = _get_run_action(run_id)
    paths = _prepare_artifact_paths(run_id)
    resume_flag = _validate_action(run_id, spec, action, paths, FORCE_RETRAIN)

    session_id = utils.make_session_id()
    eval_id = utils.make_eval_id()
    eval_output_dir = utils.make_eval_output_dir(OUTPUT_ROOT, run_id, eval_id=eval_id)
    eval_output_dir.mkdir(parents=True, exist_ok=True)

    train_dl, val_dl, test_dl, _ = create_dataloaders_for_spec(spec)
    utils.set_seed(SEED)
    model_selected, cfg_selected = build_model_for_run(run_id, spec, action, resume_flag=resume_flag)

    if action in {"train", "resume"}:
        history_selected, best_val_acc, train_time = _run_train_or_resume(
            run_id, spec, action, resume_flag,
            model_selected, cfg_selected, train_dl, val_dl, paths, session_id,
        )
    else:
        history_selected, best_val_acc, train_time = _run_reuse(
            run_id, spec, model_selected, paths
        )

    return {
        "model": model_selected,
        "cfg": cfg_selected,
        "history": history_selected,
        "best_val_acc": best_val_acc,
        "train_time": train_time,
        "spec": spec,
        "action": action,
        "eval_output_dir": eval_output_dir,
        "artifact_dir": paths["dir"],
        "best_model_path": paths["best_model"],
        "test_dl": test_dl,
    }

In [ ]:
# -- 4.2d. Vòng lặp điều phối chạy tuần tự danh sách RUN_ID_LIST --
run_results = globals().get("run_results", {})

for current_run_id in RUN_ID_LIST:
    run_results[current_run_id] = execute_single_run(current_run_id)

globals()["run_results"] = run_results
if RUN_ID_LIST:
    _last = RUN_ID_LIST[-1]
    globals()["RUN_ID"] = _last
    globals()["model_selected"] = run_results[_last]["model"]
    globals()["history_selected"] = run_results[_last]["history"]
    globals()["best_val_acc_selected"] = run_results[_last]["best_val_acc"]
    globals()["train_time_selected"] = run_results[_last]["train_time"]
    globals()["EVAL_OUTPUT_DIR"] = run_results[_last]["eval_output_dir"]

print(f"\nĐã sẵn sàng kết quả cho {len(run_results)} run: {list(run_results.keys())}")

---
## 5. So sánh Lịch sử Huấn luyện & Chi phí Mô hình (Model Comparison)

Mục này gồm 2 cell riêng biệt:
- **5.1:** Tổng hợp bảng kết quả Validation (`Best val_acc`), thời gian huấn luyện và số lượng tham số của các run đã chạy.
- **5.2:** Vẽ biểu đồ so sánh đường cong học tập (`Loss` và `Accuracy`), tự động nối tiếp trục epoch cho giai đoạn Fine-tuning.

In [ ]:
# -- 5.1. Bảng tổng hợp kết quả Validation và tham số của các Run đã chạy --
if run_results:
    rows = []
    for rid, res in run_results.items():
        p_stats = count_params(res["model"])
        rows.append({
            "Run": rid,
            "Arch": res["spec"]["arch"],
            "Action": res["action"],
            "Best val_acc": f"{res['best_val_acc']:.4f}",
            "Train time (s)": f"{res['train_time']:.1f}",
            "Total params": f"{p_stats['total']:,}",
            "Trainable params": f"{p_stats['trainable']:,}",
            "Epochs": res["spec"]["epochs"],
            "Batch": res["spec"]["batch_size"],
            "LR": res["spec"]["lr"],
        })
    df_summary = pd.DataFrame(rows).set_index("Run")
    print(df_summary.to_string())
else:
    print("Chưa có kết quả run nào. Hãy chạy Mục 4 trước.")

In [ ]:
# -- 5.2. Vẽ biểu đồ đường cong Loss & Accuracy cho các Run đã chạy --
if run_results:
    histories = {rid: res["history"] for rid, res in run_results.items() if res.get("history")}
    start_epochs = {rid: res["spec"].get("start_epoch", 0) for rid, res in run_results.items()}
    if histories:
        last_rid = list(run_results.keys())[-1]
        save_path = str(run_results[last_rid]["eval_output_dir"] / "history_comparison.png")
        evaluate.plot_history(
            histories=histories,
            start_epochs=start_epochs,
            save_path=save_path,
            show=True,
        )
    else:
        print("Không có dữ liệu history để vẽ biểu đồ.")
else:
    print("Chưa có kết quả run nào. Hãy chạy Mục 4 trước.")

---
## 6. Đánh giá Cuối cùng trên Tập Test (Final Test Evaluation)

> **Nguyên tắc chống rò rỉ dữ liệu:** Chỉ chạy sau khi hoàn tất huấn luyện và chốt checkpoint qua Validation.

Mục 6 được chia thành **7 cell đơn nhiệm** để tách biệt rõ từng khía cạnh đánh giá:
- **6.1a:** Chạy suy luận đúng 1 lượt (`run_inference`) trên tập Test và tính các độ đo phân loại (`evaluate_predictions`).
- **6.1b:** Đo độ trễ suy luận trung vị (`measure_inference_time`), tính chi phí tổng hợp (`build_cost_record`) và lưu `test_metrics.json`.
- **6.1c:** In bảng đối sánh Markdown tổng hợp tất cả các Run (`format_comparison_table`).
- **6.2:** Vẽ và lưu Ma trận nhầm lẫn (`plot_confusion_matrix`).
- **6.3a:** Thống kê và vẽ biểu đồ cột độ chính xác theo từng lớp (`per_class_accuracy` & `plot_per_class_accuracy`).
- **6.3b:** Trích xuất Top 5 cặp lớp bị nhầm lẫn nhiều nhất (`top_confusions`).
- **6.4:** Hiển thị trực quan 9 mẫu ảnh bị dự đoán sai có độ tự tin cao nhất (`misclassified_indices` & `plot_sample_predictions`).

In [ ]:
# -- 6.1a. Suy luận 1 lượt trên tập Test (run_inference) và tính Metrics phân loại --
for rid, res in run_results.items():
    model = res["model"]
    test_dl = res["test_dl"]

    model.eval()
    inf = run_inference(model, test_dl, DEVICE, loss_fn=loss_fn)
    metrics_record = evaluate_predictions(inf, CLASS_NAMES)

    res["inference_result"] = inf
    res["test_result"] = metrics_record
    res["y_true"] = inf.y_true
    res["y_pred"] = inf.y_pred

    ci_low, ci_high = metrics_record["accuracy_ci95"]
    print(
        f"[{rid:<20}] Test Loss: {metrics_record['loss']:.4f} | "
        f"Test Acc: {metrics_record['accuracy'] * 100:.2f}% "
        f"(95% CI: [{ci_low * 100:.2f}%, {ci_high * 100:.2f}%]) | "
        f"Macro F1: {metrics_record['macro_f1']:.4f}"
    )

In [ ]:
# -- 6.1b. Đo độ trễ suy luận, tính bản ghi chi phí (Cost Record) và lưu test_metrics.json --
comparison_records = {}

for rid, res in run_results.items():
    if "test_result" not in res:
        print(f"[{rid}] Chưa chạy cell 6.1a, bỏ qua.")
        continue

    model = res["model"]
    test_dl = res["test_dl"]
    spec = res["spec"]
    eval_dir = res["eval_output_dir"]
    artifact_dir = res["artifact_dir"]
    metrics_record = res["test_result"]

    sample_batch_imgs, _ = next(iter(test_dl))
    timing_info = evaluate.measure_inference_time(
        model, sample_batch_imgs, device=DEVICE, warmup=3, repeats=10
    )

    source_history = None
    dep_id = spec.get("dependency_run_id")
    if dep_id:
        if dep_id in run_results:
            source_history = run_results[dep_id]["history"]
        elif (ARTIFACT_ROOT / dep_id / "history.json").is_file():
            source_history, _ = utils.load_experiment(str(ARTIFACT_ROOT / dep_id))

    cost_record = evaluate.build_cost_record(
        num_params=count_params(model),
        model_path=str(res["best_model_path"]) if res["best_model_path"].is_file() else None,
        timing=timing_info,
        history=res["history"],
        source_history=source_history,
    )
    res["cost_record"] = cost_record

    full_test_payload = {
        "metrics": {k: v for k, v in metrics_record.items() if k != "predictions"},
        "cost": cost_record,
        "predictions": metrics_record["predictions"],
    }
    utils.save_test_metrics(str(eval_dir), full_test_payload)
    utils.save_test_metrics(str(artifact_dir), full_test_payload)

    comparison_records[rid] = {
        "metrics": metrics_record,
        "cost": cost_record,
        "best_val_acc": res["best_val_acc"],
    }
    print(
        f"[{rid:<20}] Latency: {cost_record['ms_per_image']:.2f} ms/img | "
        f"Model size: {cost_record['model_file_mb']:.1f} MB | "
        f"Saved: {eval_dir / 'test_metrics.json'}"
    )

In [ ]:
# -- 6.1c. Xuất bảng đối sánh tổng hợp tất cả các Run (Markdown Table) --
if comparison_records:
    print(evaluate.format_comparison_table(comparison_records))
else:
    print("Chưa có dữ liệu comparison_records. Hãy chạy cell 6.1a và 6.1b trước.")

In [ ]:
# -- 6.2. Tính và trực quan hóa Ma trận nhầm lẫn (Confusion Matrix) --
for rid, res in run_results.items():
    if "y_true" not in res or "y_pred" not in res:
        print(f"[{rid}] Chưa có kết quả suy luận, bỏ qua.")
        continue
    cm = compute_confusion_matrix(
        y_true=res["y_true"],
        y_pred=res["y_pred"],
        num_classes=NUM_CLASSES,
    )
    res["cm"] = cm
    evaluate.plot_confusion_matrix(
        cm,
        class_names=CLASS_NAMES,
        title=f"Confusion Matrix — {rid} (Test Set)",
        save_path=str(res["eval_output_dir"] / "confusion_matrix.png"),
        show=True,
    )

In [ ]:
# -- 6.3a. Thống kê và vẽ biểu đồ độ chính xác theo từng lớp (Per-class Accuracy) --
for rid, res in run_results.items():
    if "cm" not in res:
        if "y_true" in res and "y_pred" in res:
            res["cm"] = compute_confusion_matrix(
                y_true=res["y_true"], y_pred=res["y_pred"], num_classes=NUM_CLASSES
            )
        else:
            print(f"[{rid}] Chưa có dữ liệu dự đoán, bỏ qua.")
            continue

    class_acc = evaluate.per_class_accuracy(res["cm"])
    df_class = pd.DataFrame({
        "Class": CLASS_NAMES,
        "Accuracy": [f"{a:.3f}" for a in class_acc.tolist()],
    }).sort_values("Accuracy")

    print(f"--- [{rid}] Per-class Accuracy (Thấp -> Cao) ---")
    print(df_class.to_string(index=False))
    print()

    evaluate.plot_per_class_accuracy(
        class_acc,
        class_names=CLASS_NAMES,
        title=f"Per-class Accuracy — {rid} (Test Set)",
        save_path=str(res["eval_output_dir"] / "per_class_accuracy.png"),
        show=True,
    )

In [ ]:
# -- 6.3b. Phân tích Top 5 cặp lớp bị nhầm lẫn nhiều nhất (Top Confusions) --
for rid, res in run_results.items():
    if "cm" not in res:
        print(f"[{rid}] Chưa có confusion matrix, bỏ qua.")
        continue

    top5 = evaluate.top_confusions(res["cm"], CLASS_NAMES, k=5)
    print(f"--- [{rid}] Top 5 Cặp Nhầm Lẫn Nhiều Nhất ---")
    print(f"  {'Nhãn thật':>12} -> {'Dự đoán nhầm':<12}  Số mẫu")
    print("  " + "-" * 40)
    for true_cls, pred_cls, count in top5:
        print(f"  {true_cls:>12} -> {pred_cls:<12}  {count}")
    print()

In [ ]:
# -- 6.4. Hiển thị trực quan 9 mẫu dự đoán sai có độ tự tin (Confidence) cao nhất --
for rid, res in run_results.items():
    if "inference_result" not in res:
        print(f"[{rid}] Chưa có kết quả suy luận, bỏ qua.")
        continue

    inf = res["inference_result"]
    test_dl = res["test_dl"]
    spec = res["spec"]

    wrong_indices = evaluate.misclassified_indices(
        inf.y_true, inf.y_pred, logits=inf.logits, max_items=9
    )
    if wrong_indices.numel() == 0:
        print(f"[{rid}] Không có mẫu nào dự đoán sai.")
        continue

    mean_vals, std_vals = data_setup.get_pretrained_mean_std(
        model_name=spec["arch"],
        weights_name=TRANSFORM_WEIGHTS,
    )

    test_dataset = test_dl.dataset
    sample_imgs = [
        eval_denormalize(test_dataset[int(idx)][0], mean_vals, std_vals)
        for idx in wrong_indices
    ]
    sample_true = [int(inf.y_true[int(idx)].item()) for idx in wrong_indices]
    sample_pred = [int(inf.y_pred[int(idx)].item()) for idx in wrong_indices]

    evaluate.plot_sample_predictions(
        images=sample_imgs,
        true_labels=sample_true,
        pred_labels=sample_pred,
        class_names=CLASS_NAMES,
        ncols=3,
        save_path=str(res["eval_output_dir"] / "sample_predictions.png"),
        show=True,
    )